# AI Video Comparison Tool — Kaggle

Setup:
1. GPU: **Settings → Accelerator → GPU T4 x2** (or P100)
2. Internet: **ON** (for pip install + edge-tts)
3. Upload your assets (ảnh) vào `/kaggle/input/` hoặc mount Google Drive

In [ ]:
# Cell 1: Clone repo + set working directory
import os

REPO_DIR = '/kaggle/working/ai-video-comparison-tool'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/vinhff-ff/ai-video-comparison-tool.git {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Cell 2: Install dependencies
!pip install -q -r requirements.txt
!playwright install chromium

# Verify
import torch; print('CUDA:', torch.cuda.is_available())
import edge_tts; print('edge-tts OK')
from vieneu import Vieneu; print('vieneu OK')

## Upload Assets

Cần 6 ảnh:
- `background.jpg` — nền video
- `character.png` — nhân vật chính (nên có nền trong suốt)
- `character_confused.png` — nhân vật băn khoăn (thêm mới, optional)
- `character_cart.png` — nhân vật giỏ hàng (thêm mới, optional)
- `A.jpg` — ảnh sản phẩm A
- `B.jpg` — ảnh sản phẩm B

Upload lên tab **Data** (sidebar trái) hoặc đặt vào `assets/` bên dưới.

In [ ]:
# Cell 3: Copy assets vào thư mục assets/
# Nếu đã upload lên Kaggle Dataset → uncomment + sửa đường dẫn:
# !cp /kaggle/input/my-dataset/* assets/

# Hoặc upload trực tiếp lên tab Output rồi copy:
# !cp /kaggle/working/background.jpg assets/

import os
os.makedirs('assets', exist_ok=True)
print('Current assets:', os.listdir('assets'))

In [ ]:
# Cell 4: Verify scene JSON
import json

with open('generated/scripts/example_scene.json') as f:
    data = json.load(f)

print(f'Scenes: {len(data["scenes"])}')
for i, s in enumerate(data['scenes']):
    print(f'  [{i}] "{s["text"]}" → anim={s["animation"]}, image={s["image"]}')

In [ ]:
# Cell 5: Run pipeline — Edge TTS (free, mặc định)
import sys; sys.path.insert(0, 'src')
from pipeline import generate_video_phase2

assets = {
    'background': 'assets/background.jpg',
    'character':  'assets/character.png',
    'character_confused': 'assets/character_confused.png',  # optional
    'character_cart': 'assets/character_cart.png',          # optional
    'image_a':    'assets/A.jpg',
    'image_b':    'assets/B.jpg',
}

result = await generate_video_phase2(
    scene_json_path='generated/scripts/example_scene.json',
    assets=assets,
    run_id='edge_run_001',
    engine='edge',
)
print('DONE:', result)

In [ ]:
# Cell 6: Run pipeline — VieNeu TTS v3 Turbo (preset voice)
from pipeline import generate_video_phase2

result = await generate_video_phase2(
    scene_json_path='generated/scripts/example_scene.json',
    assets=assets,
    run_id='vieneu_preset_001',
    engine='vieneu',
    voice='Phạm Tuyên',    # hoặc 'Adam', 'Minh Đức', 'Trúc Ly'...
)
print('DONE:', result)

In [ ]:
# Cell 7: Run pipeline — VieNeu TTS v3 Turbo (voice cloning)
# Cần upload file .wav (3–8s) làm reference clip

from pipeline import generate_video_phase2

result = await generate_video_phase2(
    scene_json_path='generated/scripts/example_scene.json',
    assets=assets,
    run_id='vieneu_clone_001',
    engine='vieneu',
    ref_audio='assets/my_voice_sample.wav',  # ← thay bằng file của bạn
)
print('DONE:', result)

In [ ]:
# Cell 8: Download kết quả
from pathlib import Path

for mp4 in Path('generated/videos').glob('*_final.mp4'):
    print(f'  → {mp4.name}  ({mp4.stat().st_size / 1024:.0f} KB)')

# Click phải vào file trong Output panel → Download